In [0]:
#dbutils.library.restartPython()

In [0]:
import json

In [0]:
# config_path = dbutils.widgets.get("recon_config")
# config_df = spark.read.csv(config_path, header=True, inferSchema=True)
# table_configs = config_df.first().asDict()
# print(f"Table Configs: {table_configs}")

config_path = dbutils.widgets.get("recon_config")
config_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(config_path)
print(config_df)
#table_configs = config_df.first().asDict()
print(f"Table Configs: {table_configs}")

In [0]:
from databricks.labs.lakebridge.config import (
    DatabaseConfig,
    ReconcileConfig,
    ReconcileMetadataConfig

)
from databricks.labs.lakebridge.reconcile.recon_config import (
    Filters
)
from databricks.labs.lakebridge.reconcile.recon_config import Table
from databricks.labs.lakebridge.reconcile.trigger_recon_service import TriggerReconService
from databricks.sdk import WorkspaceClient
from databricks.labs.lakebridge import __version__
from dataclasses import dataclass

In [0]:
@dataclass
class TableRecon:
    source_schema: str
    target_catalog: str
    target_schema: str
    tables: list[Table]
    source_catalog: str | None = None

In [0]:
ws = WorkspaceClient(
    product="lakebridge",
    product_version=__version__
)

In [0]:
try:
    reconcile_config = ReconcileConfig(
        data_source=table_configs["data_source"],
        report_type=table_configs["validation"].lower(),
        secret_scope=table_configs["scope_name"],

        database_config=DatabaseConfig(
            source_catalog=table_configs["source_catalog"],
            source_schema=table_configs["source_schema"],
            target_catalog=table_configs["target_catalog"],
            target_schema=table_configs["target_schema"]
        ),

        metadata_config=ReconcileMetadataConfig(
            catalog=table_configs["target_catalog"],
            schema="lakebridge_recon"
        ))

    table_recon = TableRecon(
        source_schema=table_configs["source_schema"],
        target_catalog=table_configs["target_catalog"],
        target_schema=table_configs["target_schema"],
        tables=[
            Table(
                source_name=table_configs["source_table"],
                target_name=table_configs["target_table_name"],
                join_columns=[col.strip() for col in table_configs["primary_key"].split(",")],
                filters=Filters(
                source=f"lower({table_configs['filters_column']}) {table_configs['source_filters_condition']}",
                target=f"lower({table_configs['filters_column']}) {table_configs['target_filters_condition']}"
                )
            )
        ]
    )

    result = TriggerReconService.trigger_recon(
        ws=ws,
        spark=spark,
        table_recon=table_recon,
        reconcile_config=reconcile_config
    )
    print(f"Recon Success for {table_configs['source_table']} | Recon ID: {result.recon_id}")
except Exception as e:
    print(f"Recon Failed {e}")